# Criteria for choosing between models

_Metrics & Evaluation — measuring models, data & statistics_

**Every selection score is the same idea: reward fitting the data, then subtract a penalty for being complicated.**

Every selection criterion in this lesson is built from the same two pieces:

       score = (how badly the model fits) + (a penalty for how complex the model is)

---

This notebook is a practice scaffold for the **Criteria for choosing between models** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — statsmodels

In [ ]:
import numpy as np
import statsmodels.api as sm
from sklearn.datasets import load_diabetes
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression

X, y = load_diabetes(return_X_y=True)
x = X[:, 8].reshape(-1, 1)      # one feature (s5), the clearest U-shape
n = len(y)

# --- 1. AIC / BIC FROM SCRATCH, from the Gaussian log-likelihood ---------------
def aic_bic_from_scratch(y, yhat, k):
    n = len(y)
    rss = float(np.sum((y - yhat) ** 2))
    # maximized Gaussian log-likelihood with sigma^2 = RSS/n
    ll = -0.5 * n * (np.log(2 * np.pi) + np.log(rss / n) + 1)
    aic = -2 * ll + 2 * k
    bic = -2 * ll + k * np.log(n)
    return aic, bic

for deg in (1, 2, 3, 4, 6):
    Xd = np.hstack([x ** p for p in range(deg + 1)])   # design matrix incl. intercept
    beta, *_ = np.linalg.lstsq(Xd, y, rcond=None)
    yhat = Xd @ beta
    k = (deg + 1) + 1                                   # coefficients + variance param
    aic, bic = aic_bic_from_scratch(y, yhat, k)
    print(f"deg {deg}: from-scratch AIC={aic:.1f}  BIC={bic:.1f}")

# --- 2. statsmodels gives .aic / .bic directly ---------------------------------
Xd = sm.add_constant(np.hstack([x ** p for p in range(1, 4)]))   # cubic
ols = sm.OLS(y, Xd).fit()
print("statsmodels cubic  AIC=%.1f  BIC=%.1f" % (ols.aic, ols.bic))
# For a GLM (e.g. logistic / Poisson) the same .aic / .bic attributes exist,
# and ols.llf is the maximized log-likelihood, ols.deviance the deviance.

# --- 3. scikit-learn: cross-validated loss (assumption-light selection) ---------
for deg in (1, 2, 3, 4, 6):
    pipe = make_pipeline(StandardScaler(),
                         PolynomialFeatures(deg, include_bias=False),
                         LinearRegression())
    # negative MSE so that 'higher is better'; flip the sign to report MSE
    mse = -cross_val_score(pipe, x, y, cv=5,
                           scoring="neg_mean_squared_error").mean()
    print(f"deg {deg}: 5-fold CV MSE = {mse:.0f}")

# --- 4. Bayesian models: WAIC and PSIS-LOO via arviz ---------------------------
# After sampling a model with PyMC / Stan into an InferenceData object 'idata':
#   import arviz as az
#   az.waic(idata)              # WAIC: elpd_waic and effective params p_waic
#   az.loo(idata)              # PSIS-LOO: elpd_loo, p_loo, and Pareto-k diagnostics
#   az.compare({"m1": idata1, "m2": idata2}, ic="loo")   # rank models by ELPD

## Visualize the data & results

_As we fit polynomials of higher degree to one feature of the diabetes data, where do AIC and BIC bottom out — and where does pushing the degree higher start to OVERFIT?_

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes

d = load_diabetes()
y = d.target
x = d.data[:, 8]                       # one feature: s5 (clearest dip)
xs = (x - x.mean()) / x.std()          # standardize for a stable polynomial fit
n = len(y)

aic, bic = [], []
for deg in range(1, 9):
    # design matrix of powers 0..deg (Vandermonde), fit by least squares
    Xd = np.vander(xs, deg + 1, increasing=True)
    beta, *_ = np.linalg.lstsq(Xd, y, rcond=None)
    rss = float(np.sum((y - Xd @ beta) ** 2))
    k = (deg + 1) + 1                   # coefficients + 1 variance parameter
    # Gaussian-MLE AIC/BIC straight from RSS:
    #   -2 lnL = n*ln(RSS/n) + const ; the const cancels when comparing degrees
    fit = n * np.log(rss / n)
    aic.append(round(fit + 2 * k, 1))
    bic.append(round(fit + k * np.log(n), 1))

print("AIC:", aic)   # [3675.4, 3676.2, 3663.4, 3665.4, 3667.1, 3669.0, 3670.8, 3671.6]
print("BIC:", bic)   # [3687.7, 3692.6, 3683.9, 3689.9, 3695.7, 3701.8, 3707.6, 3712.5]
print("AIC min at degree", int(np.argmin(aic)) + 1)   # 3
print("BIC min at degree", int(np.argmin(bic)) + 1)   # 3

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You fit a model with k = 5 parameters to n = 200 points and get a maximized log-likelihood of -310. Compute its AIC and BIC, and say which criterion penalizes complexity more here.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the fit term: $-2\ln\hat{L} = -2(-310) = 620$. — _Both AIC and BIC share this same fit term; only the penalty differs._
- AIC penalty $= 2k = 2\cdot5 = 10$, so $\text{AIC} = 620 + 10 = 630$. — _AIC charges a flat 2 per parameter regardless of sample size._
- BIC penalty $= k\ln n = 5\cdot\ln 200 = 5\cdot5.298 = 26.49$, so $\text{BIC} = 620 + 26.49 = 646.5$. — _BIC's penalty grows with $\ln n$; at $n=200$ that is $5.298$ per parameter, far more than AIC's 2._

**Answer:** AIC = 630, BIC = 646.5. BIC penalizes complexity much more here because its per-parameter penalty $\ln 200 = 5.298$ is larger than AIC's flat 2. That is why BIC tends to pick sparser models than AIC, especially as $n$ grows.

</details>

**Problem 2.** A logistic regression has fitted log-likelihood -120 and a null (intercept-only) log-likelihood of -200. Compute McFadden's pseudo-R², and explain in one sentence why an ordinary R² does not apply here.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall McFadden's formula: $1 - \dfrac{\ln\hat{L}_{\text{model}}}{\ln\hat{L}_{\text{null}}}$. — _It compares the fitted model's log-likelihood to a baseline that uses only the overall rate, just as $R^2$ compares to the mean._
- Plug in: $1 - \dfrac{-120}{-200} = 1 - 0.60 = 0.40$. — _The ratio of log-likelihoods is 0.60, so the model 'explains away' 40% of the null deviance._
- Note that logistic regression predicts probabilities, not a continuous $y$, so there is no residual sum of squares to form an ordinary $R^2$. — _Pseudo-R² stands in for $R^2$ by working on the likelihood scale instead._

**Answer:** McFadden's pseudo-R² $= 1 - (-120)/(-200) = 0.40$. Ordinary R² does not apply because the response is binary — there is no residual sum of squares — so we use a likelihood-based pseudo-R² (McFadden's, or Nagelkerke's rescaled-to-1 version) instead. Values of 0.2–0.4 are considered a good fit on McFadden's scale.

</details>

**Problem 3.** Two models for the same dataset have marginal likelihoods (evidences) $p(y\mid M_1) = 4\times10^{-8}$ and $p(y\mid M_2) = 1\times10^{-8}$. Compute the Bayes factor and interpret it. Why is the evidence automatically a complexity penalty?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bayes factor $\text{BF}_{12} = p(y\mid M_1)/p(y\mid M_2) = (4\times10^{-8})/(1\times10^{-8}) = 4$. — _It is just the ratio of the two evidences — how many times more probable the data is under $M_1$._
- Read the scale: BF $\approx 4$ is 'substantial' (roughly 3–10) but not 'strong' (10+) evidence for $M_1$. — _Bayes factors are graded on conventional thresholds; 4 means $M_1$ is moderately favored, not decisively._
- Note the evidence integrates the likelihood over ALL parameter values weighted by the prior, so a model with many parameters spreads its prior thin and pays for unused flexibility. — _This 'Bayesian Occam's razor' means evidence already balances fit against complexity — no explicit parameter count is added._

**Answer:** $\text{BF}_{12} = 4/1 = 4$: substantial (but not strong) evidence favoring $M_1$. The marginal likelihood penalizes complexity automatically because it averages the likelihood over the whole prior — a more flexible model dilutes its prior probability across many parameter values, so it is rewarded only if that flexibility genuinely improves the fit. That is the built-in Occam's razor.

</details>